### Decision Tree Model 

In [1]:
import numpy as np
import pandas as pd

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, class_label=None, proba=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.class_label = class_label
        self.proba = proba

class DecisionTreeFromScratch:
    def __init__(self, max_depth=10, min_samples_leaf=10, min_samples_split=20, max_features=None, pos_weight=5):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split  = min_samples_split
        self.max_features = max_features
        self.pos_weight = pos_weight
        self.root = None
    
    def _gini(self, y):
        """Compute gini Impurity."""
        m = len(y)
        if m == 0:
            return 0
        
        p0 = np.sum(y==0) / m
        p1 = 1 - p0

        return 1 - (p0**2 + p1**2)

    def _best_split(self, X, y):
        """Find best feature and threshold to split at each node."""
        m, n = X.shape
        if m < self.min_samples_split:
            return None, None
        
        best_gini = float('inf')
        best_feature, best_threshold = None, None

        for feature in range(n):
            # thresholds = np.unique(X[:, feature])
            thresholds = np.unique(np.percentile(X[:, feature], np.linspace(0, 100, 20)))

            for threshold in thresholds: # Test midpoints
                # threshold = (thresholds[i-1] + thresholds[i]) / 2
                left_idx = X[:, feature] <= threshold
                right_idx = ~left_idx

                if sum(left_idx) < self.min_samples_leaf or sum(right_idx) < self.min_samples_leaf:
                    continue

                gini_left = self._gini(y[left_idx])
                gini_right = self._gini(y[right_idx])

                weighted_gini = (sum(left_idx)/m) * gini_left + (sum(right_idx)/m) * gini_right
                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature
                    best_threshold = threshold
        return best_feature, best_threshold

    def _build_tree(self, X, y, depth=0):
        """Recursive tree building."""
        m = len(y)

        # stopping condition.
        pos = np.sum(y == 1)
        neg = np.sum(y == 0)
        proba = pos / (pos + neg)

        if depth >= self.max_depth or m < self.min_samples_split or len(np.unique(y)) == 1:
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)
        
        feature, threshold = self._best_split(X, y)
        if feature is None: 
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)
        
        left_idx = X[:, feature] <= threshold
        right_idx = ~left_idx

        left = self._build_tree(X[left_idx], y[left_idx], depth + 1)
        right = self._build_tree(X[right_idx], y[right_idx], depth + 1)
        return Node(feature=feature, threshold=threshold, left=left, right=right)
    
    def fit(self, X, y):
        """Fit the model."""
        X = np.array(X)
        y = np.array(y).ravel()
        self.root = self._build_tree(X, y)

    def _predict_one(self, x, node):
        """Traverse tree for one sample."""
        if node.class_label is not None:
            return node.class_label
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)
        
    def predict(self, X):
        """Predict the model for a test case."""
        X = np.array(X)
        return np.array([self._predict_one(x, self.root) for x in X])

    def _predict_proba_one(self, x, node):
        """Traverse tree for one sample to get target probability."""
        if node.proba is not None:
            return node.proba
        if x[node.feature] <= node.threshold:
            return self._predict_proba_one(x, node.left)
        else:
            return self._predict_proba_one(x, node.right)
        
    def predict_proba(self, X):
        """Predict the probability of target with model for a test case."""
        X = np.array(X)
        return np.array([self._predict_proba_one(x, self.root) for x in X])

- Time Complexity for Fit: **O( tree_depth(2^D or D) * n_features * m_samples )**
- Time Complexity for Predict: **O( test_samples * depth(D) )**

- Space Complexity: **(ni_of_nodes)** Practical: n, worst case: 2^D

### optimized Decision Tree

In [3]:
import numpy as np
import pandas as pd

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, class_label=None, proba=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.class_label = class_label
        self.proba = proba

class DecisionTreeFromScratch_Opt_Time:
    def __init__(self, max_depth=10, min_samples_leaf=10, min_samples_split=20, max_features=None, pos_weight=5):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split  = min_samples_split
        self.max_features = max_features
        self.pos_weight = pos_weight
        self.root = None
    
    def _best_split(self, X, y, indices):
        m = len(indices)

        if m < self.min_samples_split:
            return None, None

        best_gini = float('inf')
        best_feature, best_threshold = None, None

        n_features = X.shape[1]

        # Feature sampling
        features = np.arange(n_features)
        if self.max_features:
            self.max_features = int(self.max_features)
            features = np.random.choice(n_features, self.max_features, replace=False)

        for feature in features:
            feature = int(feature)

            # Sort once
            sorted_idx = indices[np.argsort(X[indices, feature])]
            X_sorted = X[sorted_idx, feature]
            y_sorted = y[sorted_idx]

            # Convert labels to 0/1 (important for speed)
            y_bin = (y_sorted == 1).astype(np.int32)

            # Total counts
            total_pos = y_bin.sum()
            total_neg = m - total_pos

            # Cumulative sums → vectorized left counts
            left_pos = np.cumsum(y_bin)[:-1]
            left_neg = np.cumsum(1 - y_bin)[:-1]

            # Right counts
            right_pos = total_pos - left_pos
            right_neg = total_neg - left_neg

            # Valid splits mask
            valid = (
                (np.arange(1, m) >= self.min_samples_leaf) &
                ((m - np.arange(1, m)) >= self.min_samples_leaf) &
                (X_sorted[1:] != X_sorted[:-1])
            )

            if not np.any(valid):
                continue

            # Gini computation (vectorized)
            left_total = left_pos + left_neg
            right_total = right_pos + right_neg

            # Avoid division by zero
            left_total = np.maximum(left_total, 1)
            right_total = np.maximum(right_total, 1)

            p_left_pos = left_pos / left_total
            p_left_neg = left_neg / left_total

            p_right_pos = right_pos / right_total
            p_right_neg = right_neg / right_total

            gini_left = 1 - (p_left_pos**2 + p_left_neg**2)
            gini_right = 1 - (p_right_pos**2 + p_right_neg**2)

            # Weighted gini
            weights_left = left_total / m
            weights_right = right_total / m

            weighted_gini = weights_left * gini_left + weights_right * gini_right

            # Apply valid mask
            weighted_gini[~valid] = np.inf

            # Find best split for this feature
            idx = np.argmin(weighted_gini)
            gini = weighted_gini[idx]

            if gini < best_gini:
                best_gini = gini
                best_feature = feature
                best_threshold = (X_sorted[idx] + X_sorted[idx + 1]) / 2

            # Early stopping for already optimized gain
            if best_gini < 1e-6:
                break

        return best_feature, best_threshold

    
    # Recursive tree building (No Array Copying)
    def _build_tree(self, X, y, indices, depth=0):
        """Recursive tree building."""
        m = len(indices)
        y_subset = y[indices]

        pos = np.sum(y_subset == 1)
        neg = m - pos
        proba = pos / (pos + neg)

        # stopping condition.
        if depth >= self.max_depth or m < self.min_samples_split or len(np.unique(y_subset)) == 1:
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)
        
        feature, threshold = self._best_split(X, y, indices)
        if feature is None: 
            return Node(class_label=1 if proba >= (1/self.pos_weight) else 0, proba=proba)
        
        left_indices = indices[X[indices, feature] <= threshold]
        right_indices = indices[X[indices, feature] > threshold]

        left = self._build_tree(X, y, left_indices, depth + 1)
        right = self._build_tree(X, y, right_indices, depth + 1)
        return Node(feature=feature, threshold=threshold, left=left, right=right)
    
    # Fit
    def fit(self, X, y):
        """Fit the model."""
        X = np.array(X, dtype=np.float32)
        y = (np.array(y).ravel() == 1).astype(np.int32)
        indices = np.arange(len(y))
        self.root = self._build_tree(X, y, indices)
        return self

    def _predict_one(self, x, node):
        """Traverse tree for one sample."""
        if node.class_label is not None:
            return node.class_label
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)
        
    def predict(self, X):
        """Predict the model for a test case."""
        X = np.array(X)
        return np.array([self._predict_one(x, self.root) for x in X])

    def _predict_proba_one(self, x, node):
        """Traverse tree for one sample to get target probability."""
        if node.proba is not None:
            return node.proba
        if x[node.feature] <= node.threshold:
            return self._predict_proba_one(x, node.left)
        else:
            return self._predict_proba_one(x, node.right)
        
    def predict_proba(self, X):
        """Predict the probability of target with model for a test case."""
        X = np.array(X)
        return np.array([self._predict_proba_one(x, self.root) for x in X])

- Time Complexity: **O(n_features * n_samples)**

## Dataset - Porto Seguro Safe Driver Prediction

In [29]:
df_train = pd.read_csv('../Cleaned-Dataset/train.csv')
df_test = pd.read_csv('../Cleaned-Dataset/test.csv')


In [31]:
train_samples = int(df_train.shape[0] * 0.8)

X_train, y_train = df_train[:train_samples].drop(columns=['target']), df_train[:train_samples]['target']
X_valid, y_valid = df_train[train_samples:].drop(columns=['target']), df_train[train_samples:]['target']

test = df_test

In [32]:
# Encoding 
# Target encode only high-cardinality column
cat_cols_target = ["ps_car_11_cat"]

cat_cols_final = [col for col in X_train.columns if col.endswith("_cat")]
cat_cols_onehot = [col for col in cat_cols_final if col not in cat_cols_target]

from category_encoders import TargetEncoder, OneHotEncoder
# --- Target Encoding ---
target_encoder = TargetEncoder(cols=cat_cols_target, smoothing=0.3)

X_train = target_encoder.fit_transform(X_train, y_train)
X_valid = target_encoder.transform(X_valid)
test = target_encoder.transform(test)

# --- One Hot Encoding ---
onehot_encoder = OneHotEncoder(cols=cat_cols_onehot, use_cat_names=True)

X_train = onehot_encoder.fit_transform(X_train)
X_valid = onehot_encoder.transform(X_valid)
test = onehot_encoder.transform(test)

In [34]:
X_train = X_train.astype(float)
X_valid = X_valid.astype(float)
test = test.astype(float)

#### Model Training 

In [6]:
# --- train Model ---

model_DT = DecisionTreeFromScratch(max_depth = 10, min_samples_leaf=20, pos_weight=5, min_samples_split=50)
model_DT.fit(X_train, y_train)


In [7]:
from sklearn.metrics import roc_auc_score, precision_recall_curve

# --- validation Score ---

valid_preds = model_DT.predict_proba(X_valid)
auc = roc_auc_score(y_valid, valid_preds)

print("Validation AUC:", auc)


Validation AUC: 0.587745985407542


In [8]:
# --- Test Prediction ---

test_preds = model_DT.predict_proba(test)

# --- create submission ---

submission = pd.DataFrame({
    "id": test['id'],
    "target": test_preds.round(4)
})

submission

,id,target
0,0.0,0.0221
1,1.0,0.0221
2,2.0,0.0323
3,3.0,0.0229
4,4.0,0.0402
...,...,...
892811,1488022.0,0.0565
892812,1488023.0,0.0389
892813,1488024.0,0.0359
892814,1488025.0,0.0240


In [ ]:
submission.to_csv("../Final-Submission/DT-submission_base.csv", index=False)

## Model-2

In [8]:
# --- train Model ---

model_DT2 = DecisionTreeFromScratch(max_depth = 6, min_samples_leaf=20, pos_weight=5, min_samples_split=50)
model_DT2.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score, precision_recall_curve

# --- validation Score ---

valid_preds = model_DT2.predict_proba(X_valid)
auc = roc_auc_score(y_valid, valid_preds)

print("Validation AUC:", auc)


Validation AUC: 0.6041961964730103


### optimized Training

- `ps_ind_15` high variance, strong and centered distribution
- `ps_reg_02` high distribution around 0, long tails.
- `ps_reg_03` strong distribution, clear seperation.
- `ps_car_13` strong distribution, high variance, 
- `ps_car_15` high distribution around 3, strong left tail 

## Model trainig with optimized DT

In [36]:
# --- Train Time Optimized Model ---

model_DT2_Opt_Time = DecisionTreeFromScratch_Opt_Time(max_depth = 8, min_samples_leaf=20, pos_weight=5, min_samples_split=50, max_features=np.sqrt(df_train.shape[1]))
model_DT2_Opt_Time.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score

# --- validation Score ---
valid_preds_Opt_Time = model_DT2_Opt_Time.predict_proba(X_valid)
auc_Opt_Time = roc_auc_score(y_valid, valid_preds_Opt_Time)

print("Validation AUC:", auc_Opt_Time)


Validation AUC: 0.5879145106885977


In [37]:
# --- Train Time Optimized Model ---

model_DT2_Opt_Time2 = DecisionTreeFromScratch_Opt_Time(max_depth = 8, min_samples_leaf=20, pos_weight=5, min_samples_split=50, max_features=(0.2 * df_train.shape[1]))
model_DT2_Opt_Time2.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score

# --- validation Score ---
valid_preds_Opt_Time = model_DT2_Opt_Time2.predict_proba(X_valid)
auc_Opt_Time = roc_auc_score(y_valid, valid_preds_Opt_Time)

print("Validation AUC:", auc_Opt_Time)


Validation AUC: 0.592338926512658


In [ ]:
# # --- Test Prediction ---

# test_preds = model_DT2_Opt_Time.predict_proba(test)

# # --- create submission ---

# submission = pd.DataFrame({
#     "id": test['id'],
#     "target": test_preds.round(4)
# })

In [ ]:
# submission.to_csv("../Final-Submission/DT2-submission_opt_time_final.csv", index=False)

#### Cross-Validation 

In [39]:
X_train_cv_opt = pd.concat((X_train, X_valid), axis=0)
y_train_cv_opt = pd.concat((y_train, y_valid), axis=0)

In [44]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def cross_validate_model(model_class, X, y, n_splits=5, **model_params):
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_scores = []
    
    X = np.array(X, dtype=float)
    y = np.array(y, dtype=int)
    

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        
        model = model_class(**model_params)
        model.fit(X[train_idx], y[train_idx])
        
        valid_preds = model.predict_proba(X[valid_idx])
        auc = roc_auc_score(y[valid_idx], valid_preds)
        
        auc_scores.append(auc)
        print(f"Fold {fold} AUC: {auc:.5f}")
    
    print("\nMean AUC:", np.mean(auc_scores))
    return np.mean(auc_scores)


cross_validate_model(
    DecisionTreeFromScratch_Opt_Time,
    X_train_cv_opt,
    y_train_cv_opt,
    max_depth=8,
    min_samples_leaf=20,
    min_samples_split=50,
    max_features = (np.sqrt(df_train.shape[1]))
)


Fold 1 AUC: 0.59209
Fold 2 AUC: 0.59296
Fold 3 AUC: 0.58766
Fold 4 AUC: 0.60124
Fold 5 AUC: 0.58855

Mean AUC: 0.5924993526403398


np.float64(0.5924993526403398)

### Grid Search + CV implementation

In [45]:
X_train_gs_cv = pd.concat((X_train, X_valid), axis=0)
y_train_gs_cv = pd.concat((y_train, y_valid), axis=0)
test = test

In [46]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from itertools import product

def grid_search_cv(model_class, param_grid, X, y, n_splits=5):

    X = np.array(X, dtype=float)
    y = np.array(y, dtype=float)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Generate all param combinations
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = [dict(zip(keys, v)) for v in product(*values)]

    best_score = -np.inf
    best_params = None

    for params in combinations:
        fold_scores = []

        for train_idx, valid_idx in skf.split(X, y):
            model = model_class(**params)
            model.fit(X[train_idx], y[train_idx])

            preds = model.predict_proba(X[valid_idx])
            auc = roc_auc_score(y[valid_idx], preds)
            fold_scores.append(auc)

        mean_auc = np.mean(fold_scores)
        print(f"Params: {params} -> Mean AUC: {mean_auc:.5f}")

        if mean_auc > best_score:
            best_score = mean_auc
            best_params = params

    print("\n Best Params: ", best_params)
    print("Best CV AUC :", best_score)

    return best_params, best_score

In [47]:
n_features = X_train_gs_cv.shape[1]
n_features

109

In [48]:
param_grid = {
    "max_depth": [4, 6, 8], # [4, 6, 8]
    "min_samples_leaf": [20, 25],
    "min_samples_split": [50],
    "pos_weight": [5, 10],
    "max_features": [n_features, np.sqrt(n_features), (0.2 * n_features)]
}


In [49]:
best_params, best_auc = grid_search_cv(
    DecisionTreeFromScratch_Opt_Time, 
    param_grid,
    X_train_gs_cv,
    y_train_gs_cv,
    n_splits=5
)

print("\n Best Params: ", best_params)
print("Best CV AUC :", best_auc)

Params: {'max_depth': 4, 'min_samples_leaf': 20, 'min_samples_split': 50, 'pos_weight': 5, 'max_features': 109} -> Mean AUC: 0.59157
Params: {'max_depth': 4, 'min_samples_leaf': 20, 'min_samples_split': 50, 'pos_weight': 5, 'max_features': np.float64(10.44030650891055)} -> Mean AUC: 0.58174
Params: {'max_depth': 4, 'min_samples_leaf': 20, 'min_samples_split': 50, 'pos_weight': 5, 'max_features': 21.8} -> Mean AUC: 0.58865
Params: {'max_depth': 4, 'min_samples_leaf': 20, 'min_samples_split': 50, 'pos_weight': 10, 'max_features': 109} -> Mean AUC: 0.59157
Params: {'max_depth': 4, 'min_samples_leaf': 20, 'min_samples_split': 50, 'pos_weight': 10, 'max_features': np.float64(10.44030650891055)} -> Mean AUC: 0.57239
Params: {'max_depth': 4, 'min_samples_leaf': 20, 'min_samples_split': 50, 'pos_weight': 10, 'max_features': 21.8} -> Mean AUC: 0.58193
Params: {'max_depth': 4, 'min_samples_leaf': 25, 'min_samples_split': 50, 'pos_weight': 5, 'max_features': 109} -> Mean AUC: 0.59157
Params: {'ma

- Best Params:  {'max_depth': 6, 'min_samples_leaf': 20, 'min_samples_split': 50, 'pos_weight': 10, 'max_features': 109}
- Best CV AUC : 0.6057908961745726

##### Finally Best Params and auc score with Decision Trees are
- Best Params:  {'max_depth': 6, 'min_samples_leaf': 20, 'min_samples_split': 50, 'pos_weight': 10, 'max_features': 109}
- Best CV AUC : 0.6057908961745726


##### Best Model & Final Submission

In [50]:

best_model = DecisionTreeFromScratch_Opt_Time(max_depth=6, min_samples_leaf=20, min_samples_split=50, pos_weight=10)
best_model.fit(X_train, y_train)


In [55]:
y_preds_best = best_model.predict_proba(X_valid)

auc_best = roc_auc_score(y_valid, y_preds_best)
gini_best = 2 * auc_best - 1

print("AUC Score for Best Model: ", auc_best)
print("Gini Score for Best Model: ", gini_best)

AUC Score for Best Model:  0.6061213833104218
Gini Score for Best Model:  0.21224276662084351


In [51]:
test_preds = best_model.predict_proba(test)

best_submission = pd.DataFrame({
    'id': test['id'],
    'target': test_preds.round(4)
})

In [52]:
best_submission.to_csv('../Final-Submission/DT-submission_best.csv')

### Sklearn Model

In [53]:
from sklearn.tree import DecisionTreeClassifier

model_DT_sklearn = DecisionTreeClassifier(criterion='gini', max_depth=6, min_samples_leaf=20, min_samples_split=50)
model_DT_sklearn.fit(X_train, y_train)


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",50
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curren

In [54]:
y_preds_proba = model_DT_sklearn.predict_proba(X_valid)[:, 1]
print("AUC Score:", roc_auc_score(y_valid, y_preds_proba))


AUC Score: 0.6061213833104218
